## Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import col, trim, lower

## Read Bronze table

In [0]:
df = spark.table("workspace.bronze.crm_products")

## Silver Transformations

### Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

## Category name Cleanup

In [0]:
df = (
    df
    .withColumn(
        "product_category_name",
        F.when(col("product_category_name") == "fashio_female_clothing", "fashion_female_clothing")
         .when(col("product_category_name") == "costruction_tools_tools", "construction_tools_tools")
         .when(col("product_category_name") == "costruction_tools_garden", "construction_tools_garden")
         .when(col("product_category_name") == "home_confort", "home_comfort")
         .when(col("product_category_name") == "home_comfort_2", "home_comfort")
         .when(col("product_category_name") == "home_appliances_2", "home_appliances")
         .when(col("product_category_name").isNull(), "unknown")
         .otherwise(col("product_category_name"))
    )
)

### Remove duplicates

In [0]:
df = df.dropDuplicates(["product_id"])

### Renaming Columns

In [0]:
RENAME_MAP = {
    "product_category_name": "category_name",
    "product_weight_g": "weight_g",
    "product_length_cm": "length_cm",
    "product_height_cm": "height_cm",
    "product_width_cm": "width_cm",
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

### Sanity checks of dataframe

In [0]:
df.limit(10).display()

## Writing Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_products")

### Sanity checks of silver table

In [0]:
%sql
SELECT * FROM workspace.silver.crm_products LIMIT 10